# ARC-AGI-3 Solver — Qwen3.8-27B-FP8 — 25-Game P1 Public Eval

Public/offline evaluation is overridden to the same 25 public games × 1 pass shape. Competition reruns still use the live private game list from the Kaggle gateway.


In [ ]:
import contextlib
import json
import os
import pickle
import subprocess
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import TextIO
from urllib.request import urlopen


def _env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "").strip().lower()
    if not raw:
        return default
    return raw in {"1", "true", "yes", "y", "on"}


NOTEBOOK_START_EPOCH = time.time()
RUN_AS_SUBMISSION = False
RUN_AS_SUBMISSION = RUN_AS_SUBMISSION or _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
ENABLE_GPU = True

os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if RUN_AS_SUBMISSION else "0"
os.environ.setdefault("MPLBACKEND", "Agg")

if ENABLE_GPU:
    cuda_library_path = "/usr/local/nvidia/lib64"
    existing = [entry for entry in os.environ.get("LIBRARY_PATH", "").split(os.pathsep) if entry]
    os.environ["LIBRARY_PATH"] = os.pathsep.join(
        [cuda_library_path, *[entry for entry in existing if entry != cuda_library_path]]
    )

print(f"TAAF RUN_AS_SUBMISSION={RUN_AS_SUBMISSION}")
if ENABLE_GPU:
    print(f"taaf.kaggle: LIBRARY_PATH={os.environ['LIBRARY_PATH']}")

In [ ]:
wheelhouse = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")
if wheelhouse.exists():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-index",
            "--no-warn-conflicts",
            "--disable-pip-version-check",
            "--find-links",
            str(wheelhouse),
            "arc-agi",
        ]
    )
elif os.getenv("TAAF_KAGGLE_BUNDLE_DIR"):
    print(f"Competition wheelhouse not found at {wheelhouse}; assuming local debug dependencies are installed.")
else:
    raise RuntimeError(f"Competition wheelhouse not found at {wheelhouse}.")

In [ ]:
# Qwen3.8 / Kaggle input configuration
DATASET_SOURCES: list[str] = [
    "jakobbrggen/taaf-kaggle-source-anim-20260807-anim",
    "driessmit1/arc3-vllm-h100-wheelhouse-v3",
]
KERNEL_SOURCES: list[str] = []

# New private Kaggle Model (Version 1).
QWEN_MODEL_OWNER = "foysalemonshanto"
QWEN_MODEL_SLUG = "qwen3-8-27b-fp8-repacked-v1"
QWEN_MODEL_REF = f"{QWEN_MODEL_OWNER}/{QWEN_MODEL_SLUG}"
QWEN_MODEL_VARIATION = "hf-fp8"
QWEN_MODEL_VERSION = "1"
QWEN_SERVED_MODEL_NAME = "Qwen/Qwen3.8-27B-FP8"
QWEN_MODEL_PATH = Path(
    f"/kaggle/input/models/{QWEN_MODEL_OWNER}/{QWEN_MODEL_SLUG}/"
    f"pytorch/{QWEN_MODEL_VARIATION}/{QWEN_MODEL_VERSION}"
)

DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
WORKING_DIR = Path(os.getenv("TAAF_KAGGLE_WORKING_DIR", "/kaggle/working")).resolve()
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
SOFT_DEADLINE_BUFFER_S = 600.0
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Keep the whole run offline. vLLM/Transformers must use the mounted files only.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"


def _split_ref(ref: str) -> tuple[str, str]:
    owner, slug = ref.split("/", 1)
    return owner, slug


def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((candidate for candidate in candidates if candidate.exists()), None)


def _find_taaf_bundle() -> Path:
    explicit = os.getenv("TAAF_KAGGLE_BUNDLE_DIR", "").strip()
    if explicit:
        path = Path(explicit)
        if (path / DATASET_BUNDLE_MARKER).is_file():
            return path

    # Prefer the attached bundle whose marker actually exists.
    for root in [Path("/kaggle/input/datasets"), Path("/kaggle/input"), Path.cwd()]:
        if root.exists():
            for marker in root.rglob(DATASET_BUNDLE_MARKER):
                return marker.parent

    raise RuntimeError("Could not find TAAF Kaggle source bundle dataset.")


def _load_setup_env() -> dict[str, str]:
    if not SETUP_ENV_PATH.is_file():
        return {}
    data = json.loads(SETUP_ENV_PATH.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise RuntimeError(f"{SETUP_ENV_PATH} must contain a JSON object.")
    return {str(key): str(value) for key, value in data.items()}


def _write_setup_env_updates(updates: dict[str, str]) -> None:
    data = _load_setup_env()
    data.update(updates)
    SETUP_ENV_PATH.write_text(
        json.dumps(data, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )


BUNDLE_DIR = _find_taaf_bundle()
print(f"TAAF source bundle: {BUNDLE_DIR}")

# Verify the Qwen3.8 Kaggle Model before any expensive setup work starts.
if not QWEN_MODEL_PATH.is_dir():
    raise FileNotFoundError(
        "Qwen3.8 Kaggle Model is not attached.\n"
        f"Expected path:\n{QWEN_MODEL_PATH}\n\n"
        "Attach: Qwen3.8 27B FP8 Repacked → PyTorch → hf-fp8 → Version 1"
    )

_required_qwen_files = [
    "config.json",
    "model.safetensors.index.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "outside.safetensors",
    "mtp.safetensors",
    "chat_template.jinja",
]
_missing_qwen_files = [
    name for name in _required_qwen_files if not (QWEN_MODEL_PATH / name).is_file()
]
if _missing_qwen_files:
    raise FileNotFoundError(
        "Qwen3.8 mount is incomplete; missing: " + ", ".join(_missing_qwen_files)
    )

_qwen_layer_shards = sorted(QWEN_MODEL_PATH.glob("model-layers-*.safetensors"))
_qwen_safetensors = sorted(QWEN_MODEL_PATH.glob("*.safetensors"))
if len(_qwen_layer_shards) != 16 or len(_qwen_safetensors) != 18:
    raise RuntimeError(
        "Unexpected Qwen3.8 checkpoint layout: "
        f"{len(_qwen_layer_shards)} layer shards, "
        f"{len(_qwen_safetensors)} safetensors files."
    )

# Tell setup commands and solver code where Kaggle mounted every attached input.
kaggle_input_paths: dict[str, str] = {}
for index, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if index == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])

for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# The bundled setup resolver asks for owner/slug. Give it a model ref that maps
# directly to the full Kaggle Model version directory.
kaggle_input_paths[QWEN_MODEL_REF] = str(QWEN_MODEL_PATH)

setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
    "TAAF_QWEN_MODEL_REF": QWEN_MODEL_REF,
    "TAAF_QWEN_MODEL_PATH": str(QWEN_MODEL_PATH),
    "TAAF_QWEN_SERVED_MODEL_NAME": QWEN_SERVED_MODEL_NAME,
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
}
os.environ.update(setup_env)
_write_setup_env_updates(setup_env)

print("\n✅ Qwen3.8 input configuration ready")
print(f"Model ref:       {QWEN_MODEL_REF}")
print(f"Physical path:   {QWEN_MODEL_PATH}")
print(f"Served model:    {QWEN_SERVED_MODEL_NAME}")
print(f"Safetensors:     {len(_qwen_safetensors)}")
print(f"Layer shards:    {len(_qwen_layer_shards)}")
print(f"TAAF input map:  {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")


In [ ]:
# Audit the attached inputs that matter for this run.
print("=== TAAF bundle ===")
print(BUNDLE_DIR)
print("Exists:", BUNDLE_DIR.exists())

print("\n=== vLLM wheelhouse ===")
_vllm_wheelhouse = Path(
    "/kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3"
)
print(_vllm_wheelhouse)
print("Exists:", _vllm_wheelhouse.exists())

print("\n=== Qwen3.8 Kaggle Model ===")
print(QWEN_MODEL_PATH)
print("Exists:", QWEN_MODEL_PATH.exists())
print("Safetensors:", len(list(QWEN_MODEL_PATH.glob("*.safetensors"))))
print(
    "Repacked layer shards:",
    len(list(QWEN_MODEL_PATH.glob("model-layers-*.safetensors"))),
)


In [ ]:
import re


def _source_path_entries(bundle_dir: Path) -> list[Path]:
    src_root = bundle_dir / "src"
    if not src_root.is_dir():
        return []

    entries: list[Path] = []
    for repo in sorted(src_root.iterdir(), reverse=True):
        if not repo.is_dir():
            continue
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


def _command_env() -> dict[str, str]:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env["HF_HUB_OFFLINE"] = "1"
    env["TRANSFORMERS_OFFLINE"] = "1"
    env.update(_load_setup_env())
    return env


def _replace_python_assignment(
    command: str,
    variable_name: str,
    value: str,
) -> tuple[str, int]:
    """Replace a top-level Python string assignment inside the setup here-doc."""
    pattern = rf"(?m)^{re.escape(variable_name)}\s*=\s*(['\"])[^\r\n]*?\1\s*$"
    replacement = f"{variable_name} = {value!r}"
    return re.subn(pattern, replacement, command, count=1)


def _patch_qwen38_setup_commands(commands: list[str]) -> list[str]:
    """
    Preserve the TAAF deployment setup but replace its model identity with the
    Qwen3.8 Kaggle Model. This avoids copying/forking the large bundled setup
    script and keeps the wheelhouse/GPU/vLLM behavior from the source bundle.
    """
    patched: list[str] = []
    replacement_counts = {
        "MODEL_OWNER": 0,
        "MODEL_SLUG": 0,
        "SERVED_MODEL_NAME": 0,
    }

    replacements = {
        "MODEL_OWNER": QWEN_MODEL_OWNER,
        "MODEL_SLUG": QWEN_MODEL_SLUG,
        "SERVED_MODEL_NAME": QWEN_SERVED_MODEL_NAME,
    }

    for raw_command in commands:
        command = str(raw_command)

        for variable_name, value in replacements.items():
            command, count = _replace_python_assignment(
                command,
                variable_name,
                value,
            )
            replacement_counts[variable_name] += count

        # Make offline behavior explicit in the child process as well.
        if "def vllm_env()" in command:
            command = command.replace(
                "'VLLM_NO_USAGE_STATS': '1',",
                "'VLLM_NO_USAGE_STATS': '1',\n"
                "            'HF_HUB_OFFLINE': '1',\n"
                "            'TRANSFORMERS_OFFLINE': '1',",
                1,
            )

        patched.append(command)

    missing = [
        name for name, count in replacement_counts.items() if count == 0
    ]
    if missing:
        raise RuntimeError(
            "Could not update the bundled TAAF setup for Qwen3.8. "
            "Missing assignment(s): "
            + ", ".join(missing)
            + ". The attached TAAF bundle's setup_commands.json has changed."
        )

    print("taaf.kaggle: Qwen3.8 setup patch =", replacement_counts, flush=True)
    return patched


def _run_shell_commands(filename: str, *, label: str, check: bool) -> None:
    path = BUNDLE_DIR / filename
    if not path.is_file():
        return

    commands = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(commands, list):
        raise RuntimeError(f"{path} must contain a JSON list of shell commands.")

    if filename == "setup_commands.json":
        commands = _patch_qwen38_setup_commands(commands)

    env = _command_env()
    for command in commands:
        print(f"taaf.kaggle: {label} command: {command}", flush=True)
        result = subprocess.run(
            str(command),
            shell=True,
            check=check,
            cwd=WORKING_DIR,
            env=env,
        )
        if not check and result.returncode != 0:
            print(
                f"taaf.kaggle: {label} command exited with {result.returncode}",
                flush=True,
            )

        # Setup commands may export additional runtime settings.
        env.update(_load_setup_env())
        os.environ.update(env)


# Make bundled TAAF repos importable for this notebook and child Python processes.
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))

if source_entries:
    import sysconfig

    pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
    pth_path.write_text(
        "".join(f"{entry}\n" for entry in source_entries),
        encoding="utf-8",
    )
    print(
        f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)",
        flush=True,
    )

# Run the TAAF deployment setup, patched to use Qwen3.8.
_run_shell_commands("setup_commands.json", label="setup", check=True)

# Setup commands may export PYTHONPATH through TAAF_KAGGLE_SETUP_ENV.
pythonpath_entries = [
    entry for entry in os.environ.get("PYTHONPATH", "").split(os.pathsep) if entry
]
for entry in reversed(pythonpath_entries):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# Fail early if the analyzer is still exposing an old model identity.
_actual_model_id = os.environ.get("INFERENCE_ANALYZER_MODEL", "")
if _actual_model_id != QWEN_SERVED_MODEL_NAME:
    raise RuntimeError(
        "TAAF setup completed, but the analyzer model ID is wrong: "
        f"{_actual_model_id!r}; expected {QWEN_SERVED_MODEL_NAME!r}"
    )

print("\n✅ TAAF/vLLM setup completed for Qwen3.8")
print("Model path:", QWEN_MODEL_PATH)
print("Analyzer model:", _actual_model_id)
print("Analyzer endpoint:", os.environ.get("LOCAL_ANALYZER_BASE_URL"))


In [ ]:
# EXEC-WM PATCH CELL (prereg execwm_prereg_2026-08-25.md; arm: execwm).
# Executable world model: mine per-action object rules from recorded history,
# verify them prequentially, BFS-plan inside the verified program, and fall
# back PER LEVEL to the stock agent (the certified floor) everywhere else.
# Patch pattern: bundle copied, ONE new module written, ONE anchored solver.py
# replacement asserted count==1 -- drift dies LOUDLY, never a silent stock run.
import hashlib, shutil, sys

assert "inference" not in sys.modules, "EXECWM FATAL: inference imported before patch cell"

_EWM_SHA = 'd5c4b85872d5bba5'
_EWM_SOURCE = r'''"""exec-WM: executable world model controller for the duck harness (arm: execwm).

Mechanism class (three independent ~99-100% systems + arXiv 2605.05138): write each
game's mechanics as an executable program, VERIFY it against recorded history, and
PLAN inside it with search. Adapted for the offline Qwen3.8-27B rail, where tokens
are the binding constraint and actions are nearly free (eps=0.17):

  PHASE E  scripted exploration (deterministic, zero LLM tokens)
  PHASE I  induction: deterministic object-delta rule MINING first; the LLM is
           called only for actions mining cannot explain (budget-capped, lean
           prompt, constrained JSON fill -- never trusted unverified)
  PHASE V  mechanical prequential verification against ALL recorded transitions
           (interior-masked; lawbook-style run gating). Unverified => never used.
  PHASE P  BFS inside the verified program; the plan executes in the real game
           one action at a time with a per-step settled-frame prediction check.
           A break aborts the plan, feeds the counterexample back to the miner.
  FALLBACK per level: if no verified program or the plan budget is exhausted the
           stock ToolAgent runs UNTOUCHED for that level. Fallback == the
           certified floor behaviour. exec-WM can add levels, never remove them.

Object-centric by construction (lawbook, 647 real-27B actions: board-keyed
memoization DEAD at 2.6%/47%, global effect signatures DEAD, object-level laws
83.7-92.4% precision with run gating).

HUD strips are excluded from every comparison (P0.3: the full-grid signature
provably cannot fire; 18/25 games have a border strip ticking on >=50% of steps).

stdlib only. No LLM-router package, no vendor SDK -- raw HTTP via requests, lazily
imported, optional. Deployed into the bundle as inference/agent/exec_wm.py by the patch
cell; this file in duck_eval/execwm/ is the single source of truth.
"""
from __future__ import annotations

import json
import os
import threading
import time

EXECWM_VERSION = "v1"

# ---- sealed parameters (prereg execwm_prereg_2026-08-25.md; not tunable) ----
MOVE_ACTIONS = ("ACTION1", "ACTION2", "ACTION3", "ACTION4", "ACTION5")
E_REPEATS = 4            # times each candidate action is pressed in PHASE E
E_MAX_TURNS = 4          # controller turns PHASE E may consume per level
PROBE_BATCH = 10         # env actions executed per controller turn in PHASE E
PLAN_BATCH = 24          # plan steps executed per controller turn in PHASE P
MASK_MIN_PAIRS = 8       # frame pairs before the HUD mask converges
MASK_RATE = 0.5          # change rate for a row/col to count as HUD
MASK_BORDER = 6          # a HUD run must touch within this of the border
MAX_DELTA = 8            # translation search window (|dr|,|dc| <= MAX_DELTA)
MIN_SPRITE_CELLS = 2     # a moving component must have >= this many cells
VERIFY_MIN_N = 3         # prequential checks required before a rule is trusted
VERIFY_PRECISION = 0.90  # acceptance threshold
VERIFY_TAIL_OK = 2       # the last k checks must all be exact (run gating)
MIN_VERIFIED_MOVES = 2   # BFS needs at least this many verified move rules
MAX_BREAKS_PER_LEVEL = 3 # prediction breaks before the level falls back
MAX_GOAL_COLORS = 6      # candidate goal colors ranked by rarity
GOAL_MAX_CELLS = 30      # a goal color must be this rare on the interior
MAX_SWEEP_PLANS = 48     # coverage-sweep plans after goals are exhausted
MAX_PLANS_PER_LEVEL = 96 # total executed plans per level before fallback
LLM_CALLS_PER_GAME = 2   # PHASE I LLM budget (0 disables)
LLM_TIMEOUT_S = 180.0


def _now() -> float:
    return time.monotonic()


# ===========================================================================
# grid helpers
# ===========================================================================
def grid_of(frame_payload):
    """Normalize a runtime-state frame payload's grid to tuple-of-tuples."""
    raw = frame_payload.get("grid") if isinstance(frame_payload, dict) else None
    if not isinstance(raw, (list, tuple)):
        return ()
    return tuple(tuple(int(c) for c in row) for row in raw if isinstance(row, (list, tuple)))


def color_counts(grid):
    counts = {}
    for row in grid:
        for c in row:
            counts[c] = counts.get(c, 0) + 1
    return counts


# ===========================================================================
# HUD mask (P0.4 lineage: border-strip detection; empty before convergence)
# ===========================================================================
class HudMask:
    """Rows/cols that change on >= MASK_RATE of same-level consecutive frame
    pairs, in maximal runs touching within MASK_BORDER of the border. Degrades
    to an empty mask before MASK_MIN_PAIRS pairs have been seen."""

    def __init__(self):
        self.pairs = 0
        self._row_hits = {}
        self._col_hits = {}
        self._rows = 0
        self._cols = 0

    def observe(self, before, after):
        if not before or not after or len(before) != len(after):
            return
        self._rows = len(before)
        self._cols = max(len(r) for r in before)
        self.pairs += 1
        changed_rows, changed_cols = set(), set()
        for r, (rb, ra) in enumerate(zip(before, after)):
            if rb == ra:
                continue
            for c, (b, a) in enumerate(zip(rb, ra)):
                if b != a:
                    changed_rows.add(r)
                    changed_cols.add(c)
        for r in changed_rows:
            self._row_hits[r] = self._row_hits.get(r, 0) + 1
        for c in changed_cols:
            self._col_hits[c] = self._col_hits.get(c, 0) + 1

    def _runs(self, hits, size):
        hot = sorted(i for i, n in hits.items() if self.pairs and n / self.pairs >= MASK_RATE)
        out = set()
        run = []
        for i in hot + [None]:
            if run and (i is None or i != run[-1] + 1):
                if min(run) < MASK_BORDER or max(run) >= size - MASK_BORDER:
                    out.update(run)
                run = []
            if i is not None:
                run.append(i)
        return out

    def masked_rows(self):
        if self.pairs < MASK_MIN_PAIRS:
            return set()
        return self._runs(self._row_hits, self._rows)

    def masked_cols(self):
        if self.pairs < MASK_MIN_PAIRS:
            return set()
        return self._runs(self._col_hits, self._cols)

    def excluded(self, r, c):
        return r in self.masked_rows() or c in self.masked_cols()

    def interior_cells(self, grid):
        mr, mc = self.masked_rows(), self.masked_cols()
        for r, row in enumerate(grid):
            if r in mr:
                continue
            for c in range(len(row)):
                if c not in mc:
                    yield r, c


def interior_diff(before, after, mask: HudMask):
    """Changed interior cells between two settled frames."""
    mr, mc = mask.masked_rows(), mask.masked_cols()
    out = []
    for r, (rb, ra) in enumerate(zip(before, after)):
        if r in mr or rb == ra:
            continue
        for c, (b, a) in enumerate(zip(rb, ra)):
            if b != a and c not in mc:
                out.append((r, c))
    return out


# ===========================================================================
# transitions (rebuilt from the harness runtime-state history every turn)
# ===========================================================================
class Transition:
    __slots__ = ("action", "before", "after", "level_before", "level_after")

    def __init__(self, action, before, after, level_before, level_after):
        self.action = action
        self.before = before
        self.after = after
        self.level_before = level_before
        self.level_after = level_after


def read_state(state_path):
    """Parse tool_runtime_state.json with stdlib only. Returns (current, history)
    where current is {'grid':..., 'level':..., 'step':...} and history is a list
    of {'action', 'grid', 'level'}."""
    try:
        payload = json.loads(state_path.read_text(encoding="utf-8"))
    except Exception:
        return None, []
    cur = payload.get("current_frame") or {}
    current = {"grid": grid_of(cur), "level": int(cur.get("level", 1) or 1),
               "step": int(cur.get("step", 0) or 0)}
    history = []
    for entry in payload.get("history") or []:
        if not isinstance(entry, dict):
            continue
        fr = entry.get("frame") or {}
        history.append({"action": str(entry.get("action", "")).strip(),
                        "grid": grid_of(fr),
                        "level": int(fr.get("level", 1) or 1)})
    return current, history


# model-facing display labels <-> engine action ids (mirror of the bundle's
# inference/agent/action_names.py; history entries store the MODEL labels)
ENGINE_TO_MODEL = {"ACTION1": "UP", "ACTION2": "DOWN", "ACTION3": "LEFT",
                   "ACTION4": "RIGHT", "ACTION5": "SPACE", "ACTION6": "MOUSE",
                   "RESET": "RESET"}
MODEL_TO_ENGINE = {v: k for k, v in ENGINE_TO_MODEL.items()}


def action_token(display: str) -> str:
    """Engine action id from a history display label: 'UP' -> 'ACTION1',
    'MOUSE(row=3, col=4)' -> 'ACTION6', 'ACTION1' -> 'ACTION1'."""
    raw = display.split(" ", 1)[0].split("(", 1)[0].strip().upper()
    if raw in ENGINE_TO_MODEL:
        return raw
    return MODEL_TO_ENGINE.get(raw, raw)


def transitions_from_history(history):
    out = []
    for prev, cur in zip(history, history[1:]):
        act = action_token(cur["action"])
        if not act or not prev["grid"] or not cur["grid"]:
            continue
        out.append(Transition(act, prev["grid"], cur["grid"],
                              prev["level"], cur["level"]))
    return out


# ===========================================================================
# translation mining (PHASE I, deterministic half)
# ===========================================================================
def detect_translation(before, after, mask: HudMask):
    """Classify one interior transition.

    Returns one of
      ("noop", None)
      ("move", (dr, dc, departures:set, arrivals:set))
      ("unexplained", None)
    A move must explain EVERY interior diff cell as departure or arrival.
    """
    diff = interior_diff(before, after, mask)
    if not diff:
        return "noop", None
    diffset = set(diff)
    rows, cols = len(before), max(len(r) for r in before)
    counts = color_counts(before)
    best = None
    for dr in range(-MAX_DELTA, MAX_DELTA + 1):
        for dc in range(-MAX_DELTA, MAX_DELTA + 1):
            if dr == 0 and dc == 0:
                continue
            departures, arrivals = set(), set()
            for (r, c) in diff:
                r2, c2 = r + dr, c + dc
                if 0 <= r2 < rows and 0 <= c2 < cols and (r2, c2) in diffset \
                        and after[r2][c2] == before[r][c]:
                    departures.add((r, c))
                    arrivals.add((r2, c2))
            if len(departures) < MIN_SPRITE_CELLS:
                continue
            if diffset - departures - arrivals:
                continue
            # Symmetric ambiguity ("the background moved the other way") is
            # broken by RARITY: the true mover is the component whose colors
            # are rarest on the board (the sprite), never the background.
            rarity = sum(counts.get(before[r][c], 0) for (r, c) in departures)
            key = (rarity, abs(dr) + abs(dc), abs(dr), abs(dc))
            if best is None or key < best[0]:
                best = (key, dr, dc, departures, arrivals)
    if best is None:
        return "unexplained", None
    _, dr, dc, departures, arrivals = best
    return "move", (dr, dc, departures, arrivals)


class SpritePattern:
    """Relative cell->color pattern of the moving component, anchored at its
    top-left. Also remembers which color is rarest (the search anchor)."""

    def __init__(self, cells_to_color):
        r0 = min(r for r, _ in cells_to_color)
        c0 = min(c for _, c in cells_to_color)
        self.rel = {(r - r0, c - c0): col for (r, c), col in cells_to_color.items()}
        self.colors = frozenset(self.rel.values())
        counts = {}
        for col in self.rel.values():
            counts[col] = counts.get(col, 0) + 1
        self.anchor_color = min(counts, key=lambda k: (counts[k], k))
        self.anchor_offsets = [rc for rc, col in self.rel.items() if col == self.anchor_color]

    def find(self, grid):
        """All positions (top-left) where the full pattern matches. Uses the
        rarest color as anchor to keep the scan cheap."""
        rows, cols = len(grid), max((len(r) for r in grid), default=0)
        hits = []
        seen = set()
        for r in range(rows):
            row = grid[r]
            for c in range(len(row)):
                if row[c] != self.anchor_color:
                    continue
                for (ar, ac) in self.anchor_offsets:
                    pr, pc = r - ar, c - ac
                    if (pr, pc) in seen:
                        continue
                    seen.add((pr, pc))
                    ok = True
                    for (dr, dc), col in self.rel.items():
                        rr, cc = pr + dr, pc + dc
                        if not (0 <= rr < rows and 0 <= cc < len(grid[rr])) or grid[rr][cc] != col:
                            ok = False
                            break
                    if ok:
                        hits.append((pr, pc))
        return sorted(set(hits))


class Rule:
    __slots__ = ("kind", "delta", "n", "ok", "tail", "verified")

    def __init__(self, kind, delta=None):
        self.kind = kind          # "noop" | "move"
        self.delta = delta        # (dr, dc) for "move"
        self.n = 0                # prequential checks
        self.ok = 0               # exact matches
        self.tail = 0             # consecutive exact matches (run gating)
        self.verified = False

    def precision(self):
        return self.ok / self.n if self.n else 0.0

    def as_dict(self):
        return {"kind": self.kind, "delta": self.delta, "n": self.n,
                "ok": self.ok, "precision": round(self.precision(), 4),
                "verified": self.verified}


class WorldModel:
    """The per-level executable program: sprite pattern + per-action rules +
    learned underlay + permeable-color set. Everything mechanical."""

    def __init__(self):
        self.sprite: SpritePattern | None = None
        self.rules: dict[str, Rule] = {}
        self.underlay: dict[tuple, int] = {}      # cell -> revealed color
        self.permeable: set[int] = set()          # colors the sprite overwrote
        self.blockers: set[int] = set()           # colors observed to refuse a move
        self.unexplained = 0                       # instances the miner cannot explain
        self.mined_from = 0                        # transitions consumed

    # ---- induction (deterministic) ----
    def mine(self, transitions, mask: HudMask, llm_hints=None):
        per_action = {}
        order = []           # classification per transition, in replay order
        pattern_votes = {}   # normalized rel-pattern -> [count, rarity, cells]
        for t in transitions:
            if t.level_before != t.level_after or t.action == "RESET":
                continue
            if t.action not in MOVE_ACTIONS:
                continue
            kind, info = detect_translation(t.before, t.after, mask)
            per_action.setdefault(t.action, []).append((kind, info, t, len(order)))
            order.append(kind)
            if kind == "move":
                dr, dc, departures, arrivals = info
                counts = color_counts(t.before)
                cells = {cell: t.before[cell[0]][cell[1]] for cell in departures}
                rarity = sum(counts.get(v, 0) for v in cells.values())
                r0 = min(r for r, _ in cells)
                c0 = min(c for _, c in cells)
                key = frozenset(((r - r0, c - c0), col)
                                for (r, c), col in cells.items())
                vote = pattern_votes.setdefault(key, [0, rarity, cells])
                vote[0] += 1
                for cell in departures - arrivals:
                    self.underlay[cell] = t.after[cell[0]][cell[1]]
                for cell in arrivals - departures:
                    self.permeable.add(t.before[cell[0]][cell[1]])
        if pattern_votes:
            # CONSENSUS: the sprite is the pattern that recurs across the most
            # move instances; rarity only breaks ties. A one-off translation
            # artifact (an animation, a pickup) can never outvote the sprite.
            _, _, cells = max(pattern_votes.values(),
                              key=lambda v: (v[0], -v[1]))
            self.sprite = SpritePattern(cells)
        # An unexplained event (pickup, toggle, spawn) invalidates blocker
        # evidence recorded before it -- the world is stateful and a door can
        # open. Blockers therefore re-mine from POST-EVENT no-ops only.
        last_event = max((i for i, k in enumerate(order) if k == "unexplained"),
                         default=-1)
        self.blockers = set()
        # blocker mining: an action with move consensus that ALSO produced
        # no-ops reveals which colors refuse the move -- the colors standing in
        # the would-be destination footprint (minus known-permeable ones).
        if self.sprite is not None:
            for action, instances in per_action.items():
                deltas = {info[:2] for k, info, _, _ in instances if k == "move"}
                if len(deltas) != 1:
                    continue
                dr, dc = next(iter(deltas))
                for k, _info, t, idx in instances:
                    if k != "noop" or idx <= last_event:
                        continue
                    hits = self.sprite.find(t.before)
                    if len(hits) != 1:
                        continue
                    pr, pc = hits[0][0] + dr, hits[0][1] + dc
                    cur = {(hits[0][0] + rr, hits[0][1] + cc)
                           for (rr, cc) in self.sprite.rel}
                    nonperm = set()
                    off_board = False
                    for (rr, cc) in self.sprite.rel:
                        r2, c2 = pr + rr, pc + cc
                        if (r2, c2) in cur:
                            continue
                        if 0 <= r2 < len(t.before) and 0 <= c2 < len(t.before[r2]):
                            col = t.before[r2][c2]
                            if col not in self.permeable:
                                nonperm.add(col)
                        else:
                            off_board = True
                    # credit assignment must be UNAMBIGUOUS: only a footprint
                    # whose non-permeable colors are a singleton names its
                    # blocker (a {wall, lane} footprint blames neither).
                    if len(nonperm) == 1 and not off_board:
                        self.blockers.add(next(iter(nonperm)))
        self.rules = {}
        for action, instances in per_action.items():
            kinds = [k for k, _, _, _ in instances]
            moves = [info for k, info, _, _ in instances if k == "move"]
            if moves:
                deltas = {(dr, dc) for dr, dc, _, _ in moves}
                if len(deltas) == 1:
                    self.rules[action] = Rule("move", next(iter(deltas)))
                continue
            if kinds and all(k == "noop" for k in kinds):
                self.rules[action] = Rule("noop")
        # LLM-filled hypotheses enter here on the same footing -- as CANDIDATES
        # that PHASE V must pass before use. Mining always wins a conflict.
        for action, hint in (llm_hints or {}).items():
            if action in self.rules or action not in MOVE_ACTIONS:
                continue
            if isinstance(hint, dict) and hint.get("type") == "translate":
                try:
                    dr, dc = int(hint["dr"]), int(hint["dc"])
                except Exception:
                    continue
                if (dr or dc) and abs(dr) <= MAX_DELTA and abs(dc) <= MAX_DELTA:
                    self.rules[action] = Rule("move", (dr, dc))
        self.blockers -= self.permeable
        self.unexplained = sum(1 for k in order if k == "unexplained")
        self.mined_from = len(transitions)
        return self.rules

    # ---- prediction (used by V and by P's per-step check) ----
    def occupancy(self, grid, pos):
        """Static view of the board with the sprite removed (underlay where
        known, else None = unknown)."""
        cells = {}
        for (dr, dc) in self.sprite.rel:
            cell = (pos[0] + dr, pos[1] + dc)
            cells[cell] = self.underlay.get(cell)
        return cells

    def predict(self, grid, pos, action, mask: HudMask):
        """Predict (next_pos, predicted_cells) for a verified/candidate rule.
        predicted_cells maps cell -> color for every cell whose post-action
        value the model claims to know; None values mean 'unknown, don't
        check'. Returns None if the model cannot predict this action."""
        rule = self.rules.get(action)
        if rule is None or self.sprite is None or pos is None:
            return None
        if rule.kind == "noop":
            return pos, {}
        dr, dc = rule.delta
        rows, cols = len(grid), max(len(r) for r in grid)
        new_pos = (pos[0] + dr, pos[1] + dc)
        target_cells = {}
        for (rr, cc), col in self.sprite.rel.items():
            r2, c2 = new_pos[0] + rr, new_pos[1] + cc
            if not (0 <= r2 < rows and 0 <= c2 < cols):
                return pos, {}  # off-board => predict blocked no-op
            target_cells[(r2, c2)] = col
        current_cells = {(pos[0] + rr, pos[1] + cc) for (rr, cc) in self.sprite.rel}
        blocked = False
        for cell in target_cells:
            if cell in current_cells:
                continue
            col = grid[cell[0]][cell[1]]
            if col not in self.permeable:
                blocked = True
                break
        if blocked:
            return pos, {}      # predict exact no-op
        pred = {}
        for cell, col in target_cells.items():
            pred[cell] = col
        for cell in current_cells - set(target_cells):
            pred[cell] = self.underlay.get(cell)   # None = unknown, don't check
        return new_pos, pred

    def check_prediction(self, before, after, pos, action, mask: HudMask):
        """Compare a prediction against the settled real frame. Returns
        (exact:bool, definite_checked:int, new_pos) or None if no prediction."""
        out = self.predict(before, pos, action, mask)
        if out is None:
            return None
        new_pos, pred = out
        checked = 0
        exact = True
        for r, c in mask.interior_cells(after):
            want = pred.get((r, c), before[r][c] if r < len(before) and c < len(before[r]) else None)
            if want is None:
                # unknown underlay: learn it instead of judging it
                if (r, c) in pred:
                    self.underlay[(r, c)] = after[r][c]
                continue
            checked += 1
            if after[r][c] != want:
                exact = False
        return exact, checked, new_pos

    # ---- verification (PHASE V) ----
    def verify(self, transitions, mask: HudMask):
        """Prequential replay over the recorded history. Resets and refills
        every rule's counters; sets rule.verified by threshold + run gating."""
        for rule in self.rules.values():
            rule.n = rule.ok = rule.tail = 0
            rule.verified = False
        if self.sprite is None:
            return {}
        for t in transitions:
            if t.level_before != t.level_after or t.action not in self.rules:
                continue
            hits = self.sprite.find(t.before)
            if len(hits) != 1:
                continue
            res = self.check_prediction(t.before, t.after, hits[0], t.action, mask)
            if res is None:
                continue
            exact, checked, _ = res
            rule = self.rules[t.action]
            if not exact:
                # A miss that the miner itself cannot explain as any clean
                # translation/no-op is a WORLD EVENT (pickup, teleport,
                # spawn), not evidence against the rule -- it neither
                # confirms nor refutes, so it stays out of the count. A miss
                # that IS a clean noop/other-translation is a real failure.
                kind, _ = detect_translation(t.before, t.after, mask)
                if kind == "unexplained":
                    continue
            rule.n += 1
            if exact:
                rule.ok += 1
                rule.tail += 1
            else:
                rule.tail = 0
        for rule in self.rules.values():
            # Acceptance = sample size + precision. A consecutive-tail demand
            # was tried and REJECTED on the real ls20 rail: the game's ~1/43
            # counter-teleport made one terminal anomaly permanently kill a
            # 30-for-31 rule and halve the board's connectivity. Rare model
            # noise is priced by the precision threshold and by the per-level
            # break budget, not by a sudden-death tail.
            rule.verified = (rule.n >= VERIFY_MIN_N
                            and rule.precision() >= VERIFY_PRECISION)
        return {a: r.as_dict() for a, r in self.rules.items()}

    def verified_moves(self):
        return {a: r for a, r in self.rules.items()
                if r.verified and r.kind == "move"}


# ===========================================================================
# planner (PHASE P): BFS over sprite positions inside the verified program
# ===========================================================================
def bfs_reachable(model: WorldModel, grid, start):
    """BFS over sprite positions using only VERIFIED move rules and the
    conservative permeability model. Returns {pos: (prev_pos, action)}."""
    moves = model.verified_moves()
    sprite = model.sprite
    rows, cols = len(grid), max(len(r) for r in grid)
    parents = {start: (None, None)}
    frontier = [start]
    while frontier:
        nxt = []
        for pos in frontier:
            cur_cells = {(pos[0] + rr, pos[1] + cc) for (rr, cc) in sprite.rel}
            for action, rule in moves.items():
                dr, dc = rule.delta
                np_ = (pos[0] + dr, pos[1] + dc)
                if np_ in parents:
                    continue
                ok = True
                for (rr, cc) in sprite.rel:
                    r2, c2 = np_[0] + rr, np_[1] + cc
                    if not (0 <= r2 < rows and 0 <= c2 < cols):
                        ok = False
                        break
                    if (r2, c2) in cur_cells:
                        continue
                    col = model.underlay.get((r2, c2), grid[r2][c2])
                    # cells currently under the sprite footprint use underlay
                    if col not in model.permeable:
                        ok = False
                        break
                if ok:
                    parents[np_] = (pos, action)
                    nxt.append(np_)
        frontier = nxt
    return parents


def path_to(parents, goal):
    if goal not in parents:
        return None
    actions = []
    pos = goal
    while parents[pos][0] is not None:
        prev, action = parents[pos]
        actions.append(action)
        pos = prev
    actions.reverse()
    return actions


def goal_targets(model: WorldModel, grid, mask: HudMask, pos):
    """Ranked candidate goal positions: for each rare interior color, the
    reachable sprite positions whose footprint touches (or lands adjacent to)
    a cell of that color. Rarest colors first, then nearest cell."""
    counts = {}
    mr, mc = mask.masked_rows(), mask.masked_cols()
    cur_cells = {(pos[0] + rr, pos[1] + cc) for (rr, cc) in model.sprite.rel}
    for r, row in enumerate(grid):
        if r in mr:
            continue
        for c in range(len(row)):
            if c in mc or (r, c) in cur_cells:
                continue
            counts.setdefault(row[c], []).append((r, c))
    bg = max(counts, key=lambda k: len(counts[k])) if counts else None
    boring = {bg} | model.sprite.colors | model.permeable | set(model.underlay.values())
    ranked = sorted((k for k in counts
                     if k not in boring and len(counts[k]) <= GOAL_MAX_CELLS),
                    key=lambda k: (len(counts[k]), k))
    out = []
    for color in ranked[:MAX_GOAL_COLORS]:
        out.append((color, counts[color]))
    return out


def frontier_probes(model: WorldModel, grid, parents):
    """Cheap experiments that EXPAND the verified model: from a reachable
    position, one move whose destination footprint contains colors of unknown
    permeability (not yet walked on, not yet observed to block). Executing it
    either unlocks a new region (the miner adds the color to `permeable`) or
    records a blocker -- both are progress, and both cost one action."""
    rows, cols = len(grid), max(len(r) for r in grid)
    moves = model.verified_moves()
    out = []
    for pos in parents:
        cur = {(pos[0] + rr, pos[1] + cc) for (rr, cc) in model.sprite.rel}
        for action, rule in moves.items():
            dr, dc = rule.delta
            np_ = (pos[0] + dr, pos[1] + dc)
            if np_ in parents:
                continue
            unknown = set()
            ok = True
            for (rr, cc) in model.sprite.rel:
                r2, c2 = np_[0] + rr, np_[1] + cc
                if not (0 <= r2 < rows and 0 <= c2 < cols):
                    ok = False
                    break
                if (r2, c2) in cur:
                    continue
                col = model.underlay.get((r2, c2), grid[r2][c2])
                if col in model.permeable:
                    continue
                if col in model.blockers:
                    ok = False
                    break
                unknown.add(col)
            if ok and unknown:
                out.append((len(unknown), pos, action, tuple(sorted(unknown))))
    counts = color_counts(grid)
    # rarest unknown color first: a 12-cell door outranks a 2600-cell wall
    out.sort(key=lambda x: (min(counts.get(c, 0) for c in x[3]), x[0], x[1], x[2]))
    return out


def positions_touching(model: WorldModel, cells, parents, dilate=1):
    """Reachable sprite positions whose footprint covers any of `cells` (or a
    cell within `dilate` of one -- goals of non-permeable colors can only be
    stood NEXT TO), nearest (by BFS insertion order) first."""
    cellset = set(cells)
    if dilate:
        for (r, c) in list(cellset):
            for dr in range(-dilate, dilate + 1):
                for dc in range(-dilate, dilate + 1):
                    cellset.add((r + dr, c + dc))
    touching = []
    for p in parents:
        for (rr, cc) in model.sprite.rel:
            if (p[0] + rr, p[1] + cc) in cellset:
                touching.append(p)
                break
    return touching


# ===========================================================================
# PHASE I, LLM half (optional, budget-capped, verified before use)
# ===========================================================================
def llm_fill(unexplained, evidence_lines, llm_cfg):
    """One lean chat call asking for constrained JSON rules for the actions
    mining could not explain. Any failure returns {}. Never raises."""
    if not llm_cfg or not unexplained:
        return {}, 0
    try:
        import requests  # lazy; the bundle already depends on it
    except Exception:
        return {}, 0
    prompt = (
        "Recorded effects per action in a 64x64 grid game:\n"
        + "\n".join(evidence_lines[:24])
        + "\nFor each of these actions, answer what it does: "
        + ", ".join(sorted(unexplained))
        + '\nReply ONLY with JSON like {"ACTION3": {"type": "translate", "dr": 0, "dc": 1}} '
          'or {"ACTION3": {"type": "unknown"}}.'
    )
    try:
        resp = requests.post(
            llm_cfg["base_url"].rstrip("/") + "/chat/completions",
            headers={"Authorization": f"Bearer {llm_cfg.get('api_key') or 'none'}",
                     "Content-Type": "application/json"},
            json={"model": llm_cfg["model"], "temperature": 0.0, "max_tokens": 512,
                  "messages": [{"role": "user", "content": prompt}]},
            timeout=LLM_TIMEOUT_S,
        )
        data = resp.json()
        text = (data.get("choices") or [{}])[0].get("message", {}).get("content") or ""
        tokens = int((data.get("usage") or {}).get("completion_tokens") or 0)
        start, end = text.find("{"), text.rfind("}")
        if start < 0 or end <= start:
            return {}, tokens
        parsed = json.loads(text[start:end + 1])
        return (parsed if isinstance(parsed, dict) else {}), tokens
    except Exception:
        return {}, 0


# ===========================================================================
# the per-game controller
# ===========================================================================
class _LevelState:
    def __init__(self, level):
        self.level = level
        self.phase = "E"
        self.e_turns = 0
        self.probe_queue = []
        self.probes_done = 0
        self.model = WorldModel()
        self.plans_run = 0
        self.novelty_run = 0
        self.breaks = 0
        self.tried_targets = []
        self.visited = set()      # sprite positions already stood on
        self.events_mark = -1     # unexplained+breaks count at last probe re-arm
        self.refused_probes = []  # probe keys that came back as no-ops (locked doors)
        self.pending_probe = None # key of the probe currently executing
        self.active_plan = None   # (desc, remaining_actions) of a truncated plan
        self.fallback = False
        self.fallback_reason = ""
        self.cleared_via = None
        self.rules_report = {}


class TurnResult:
    """Duck-typed stand-in for AnalyzerTurnResult (the solver only reads these
    attributes)."""
    def __init__(self, step_executed, retryable_failure=False, reasoning="",
                 yielded_control=False):
        self.step_executed = step_executed
        self.retryable_failure = retryable_failure
        self.reasoning = reasoning
        self.yielded_control = yielded_control


class ExecWMController:
    def __init__(self, game_id, log, report_cb, llm_cfg=None):
        self.game_id = game_id
        self.log = log
        self.report_cb = report_cb
        self.llm_cfg = llm_cfg
        self.mask = HudMask()
        self.mask_fed = 0
        self.levels: dict[int, _LevelState] = {}
        self.llm_calls = 0
        self.llm_tokens = 0
        self.actions_executed = 0
        self.disabled_reason = None   # game-level triage (e.g. mouse-only)

    # ------------------------------------------------------------------
    def _feed_mask(self, history):
        for i in range(max(1, self.mask_fed), len(history)):
            prev, cur = history[i - 1], history[i]
            if prev["level"] == cur["level"] and action_token(cur["action"]) != "RESET":
                self.mask.observe(prev["grid"], cur["grid"])
        self.mask_fed = len(history)

    def _lvl(self, level) -> _LevelState:
        if level not in self.levels:
            self.levels[level] = _LevelState(level)
            self.log(f"level={level} begin phase=E")
        return self.levels[level]

    def _move_candidates(self, valid_actions):
        va = set()
        for a in (valid_actions or []):
            raw = str(a).strip().upper()
            va.add(MODEL_TO_ENGINE.get(raw, raw))
        return [a for a in MOVE_ACTIONS if a in va]

    # ------------------------------------------------------------------
    def report(self):
        return {
            "version": EXECWM_VERSION,
            "game_id": self.game_id,
            "armed": True,
            "disabled_reason": self.disabled_reason,
            "llm_calls": self.llm_calls,
            "llm_tokens": self.llm_tokens,
            "actions_executed": self.actions_executed,
            "mask_rows": sorted(self.mask.masked_rows()),
            "mask_cols": sorted(self.mask.masked_cols()),
            "levels": {
                str(l.level): {
                    "phase": l.phase,
                    "probes": l.probes_done,
                    "plans_run": l.plans_run,
                    "breaks": l.breaks,
                    "fallback": l.fallback,
                    "fallback_reason": l.fallback_reason,
                    "cleared_via": l.cleared_via,
                    "rules": l.rules_report,
                } for l in self.levels.values()
            },
        }

    # ------------------------------------------------------------------
    def wants_turn(self, current, valid_actions):
        """Does exec-WM claim this turn, or should the stock agent run?"""
        if self.disabled_reason:
            return False
        cands = self._move_candidates(valid_actions)
        if not cands:
            self.disabled_reason = "no-keyboard-actions"
            self.log(f"disabled reason={self.disabled_reason} valid={sorted(valid_actions or [])}")
            return False
        lvl = self._lvl(current["level"])
        return not lvl.fallback

    def _mark_fallback(self, lvl: _LevelState, reason):
        lvl.fallback = True
        lvl.fallback_reason = reason
        lvl.phase = "F"
        self.log(f"level={lvl.level} fallback reason={reason}")

    # ------------------------------------------------------------------
    # a controller turn: execute deterministic work through step_env
    # ------------------------------------------------------------------
    def run_turn(self, state_path, step_env, should_stop, valid_actions):
        current, history = read_state(state_path)
        if current is None or not current["grid"]:
            return TurnResult(step_executed=False)
        self._feed_mask(history)
        lvl = self._lvl(current["level"])
        transitions = [t for t in transitions_from_history(history)
                       if t.level_before == lvl.level]
        cands = self._move_candidates(valid_actions)

        if lvl.phase == "E":
            return self._turn_explore(lvl, cands, state_path, step_env, should_stop)
        if lvl.phase == "P":
            return self._turn_plan(lvl, current, transitions, state_path, step_env, should_stop)
        return TurnResult(step_executed=False)

    # ---- PHASE E ----
    def _turn_explore(self, lvl, cands, state_path, step_env, should_stop):
        if not lvl.probe_queue:
            need = []
            for rep in range(E_REPEATS):
                for a in cands:
                    need.append(a)
            lvl.probe_queue = need[lvl.probes_done:]
        lvl.e_turns += 1
        executed = 0
        while lvl.probe_queue and executed < PROBE_BATCH:
            if should_stop and should_stop():
                break
            action = lvl.probe_queue.pop(0)
            payload = step_env({"action": action})
            executed += 1
            lvl.probes_done += 1
            self.actions_executed += 1
            if not payload.get("executed"):
                continue
            if payload.get("level_completed") or payload.get("run_complete"):
                lvl.cleared_via = "explore"
                self.log(f"level={lvl.level} CLEARED via=explore probes={lvl.probes_done}")
                self._flush(state_path)
                return TurnResult(step_executed=True)
            if payload.get("game_over"):
                self.log(f"level={lvl.level} game_over during explore (auto-reset)")
                self._flush(state_path)
                return TurnResult(step_executed=True)
        self.log(f"level={lvl.level} explore turn={lvl.e_turns} probes={lvl.probes_done} "
                 f"mask_pairs={self.mask.pairs}")
        if not lvl.probe_queue:
            self._induce(lvl, state_path)
        elif lvl.e_turns >= E_MAX_TURNS:
            self._mark_fallback(lvl, "explore-budget-exhausted")
        self._flush(state_path)
        return TurnResult(step_executed=executed > 0)

    # ---- PHASE I + V ----
    def _induce(self, lvl, state_path):
        current, history = read_state(state_path)
        transitions = [t for t in transitions_from_history(history)
                       if t.level_before == lvl.level]
        lvl.model.mine(transitions, self.mask)
        lvl.rules_report = lvl.model.verify(transitions, self.mask)
        mined = sorted(lvl.model.rules)
        unexplained = [a for a in self._probe_actions(transitions) if a not in lvl.model.rules]
        self.log(f"level={lvl.level} induce mined={mined} unexplained={unexplained} "
                 f"rules={json.dumps(lvl.rules_report, sort_keys=True)}")
        if unexplained and self.llm_cfg and self.llm_calls < LLM_CALLS_PER_GAME:
            evidence = self._evidence_lines(transitions)
            hints, tokens = llm_fill(unexplained, evidence, self.llm_cfg)
            self.llm_calls += 1
            self.llm_tokens += tokens
            if hints:
                lvl.model.mine(transitions, self.mask, llm_hints=hints)
                lvl.rules_report = lvl.model.verify(transitions, self.mask)
                self.log(f"level={lvl.level} induce llm_hints={sorted(hints)} "
                         f"rules={json.dumps(lvl.rules_report, sort_keys=True)}")
        n_moves = len(lvl.model.verified_moves())
        if n_moves >= MIN_VERIFIED_MOVES and lvl.model.sprite is not None:
            lvl.phase = "P"
            self.log(f"level={lvl.level} VERIFIED moves={n_moves} -> plan "
                     f"permeable={sorted(lvl.model.permeable)} "
                     f"blockers={sorted(lvl.model.blockers)}")
        else:
            self._mark_fallback(lvl, f"no-verified-model(moves={n_moves})")

    def _probe_actions(self, transitions):
        return sorted({t.action for t in transitions if t.action in MOVE_ACTIONS})

    def _evidence_lines(self, transitions):
        lines = []
        for t in transitions[-24:]:
            diff = interior_diff(t.before, t.after, self.mask)
            if not diff:
                lines.append(f"{t.action}: no visible change")
            else:
                r0 = min(r for r, _ in diff); r1 = max(r for r, _ in diff)
                c0 = min(c for _, c in diff); c1 = max(c for _, c in diff)
                lines.append(f"{t.action}: {len(diff)} cells changed in rows {r0}-{r1} cols {c0}-{c1}")
        return lines

    # ---- PHASE P ----
    def _turn_plan(self, lvl, current, transitions, state_path, step_env, should_stop):
        grid = current["grid"]
        model = lvl.model
        hits = model.sprite.find(grid) if model.sprite else []
        if len(hits) != 1:
            lvl.breaks += 1
            self.log(f"level={lvl.level} sprite-lost hits={len(hits)} breaks={lvl.breaks}")
            if lvl.breaks >= MAX_BREAKS_PER_LEVEL:
                self._mark_fallback(lvl, "sprite-lost")
            else:
                lvl.phase = "E"
                lvl.probe_queue = list(self._move_candidates(MOVE_ACTIONS))[:2]
            self._flush(state_path)
            return TurnResult(step_executed=False)
        pos = hits[0]
        lvl.visited.add(pos)
        if lvl.active_plan is not None:
            target_desc, actions, probe_last = lvl.active_plan
            lvl.active_plan = None
        else:
            parents = bfs_reachable(model, grid, pos)
            plan = self._next_plan(lvl, grid, pos, parents)
            if plan is None:
                probes = frontier_probes(model, grid, parents)
                self.log(f"level={lvl.level} exhausted-diag parents={len(parents)} "
                         f"probes={len(probes)} tried={len(lvl.tried_targets)} "
                         f"moves={sorted(model.verified_moves())} "
                         f"first_probes={[(p[1], p[2], p[3]) for p in probes[:4]]}")
                self._mark_fallback(lvl, "plan-targets-exhausted")
                self._flush(state_path)
                return TurnResult(step_executed=False)
            target_desc, actions, probe_last = plan
            lvl.plans_run += 1
            self.log(f"level={lvl.level} plan#{lvl.plans_run} target={target_desc} "
                     f"len={len(actions)}")
        executed = 0
        clean = True
        for step_i, action in enumerate(actions[:PLAN_BATCH]):
            is_probe_step = probe_last and step_i == len(actions) - 1
            if should_stop and should_stop():
                break
            cur_state, _ = read_state(state_path)
            before = cur_state["grid"]
            bhits = model.sprite.find(before)
            bpos = bhits[0] if len(bhits) == 1 else None
            payload = step_env({"action": action})
            executed += 1
            self.actions_executed += 1
            if not payload.get("executed"):
                lvl.breaks += 1
                clean = False
                self.log(f"level={lvl.level} plan-step-refused action={action} breaks={lvl.breaks}")
                break
            if payload.get("level_completed") or payload.get("run_complete"):
                lvl.cleared_via = "plan"
                self.log(f"level={lvl.level} CLEARED via=plan plan#{lvl.plans_run} "
                         f"actions={self.actions_executed}")
                self._flush(state_path)
                return TurnResult(step_executed=True)
            if payload.get("game_over"):
                self.log(f"level={lvl.level} game_over during plan (auto-reset)")
                self._flush(state_path)
                return TurnResult(step_executed=True)
            after_state, _ = read_state(state_path)
            after = after_state["grid"]
            if bpos is not None and not is_probe_step:
                res = model.check_prediction(before, after, bpos, action, self.mask)
                if res is not None and res[0]:
                    lvl.visited.add(res[2])
                if res is not None and not res[0]:
                    clean = False
                    kind, _ = detect_translation(before, after, self.mask)
                    if kind == "unexplained":
                        # a world event, not a model error: no break charged;
                        # it feeds the event counter that re-arms probes.
                        self.log(f"level={lvl.level} EVENT action={action} "
                                 f"step={executed} (plan aborted, no break)")
                    else:
                        lvl.breaks += 1
                        self.log(f"level={lvl.level} BREAK action={action} "
                                 f"step={executed} breaks={lvl.breaks}")
                    # counterexample feeds the next mine cycle
                    cur, history = read_state(state_path)
                    self._feed_mask(history)
                    transitions = [t for t in transitions_from_history(history)
                                   if t.level_before == lvl.level]
                    lvl.model.mine(transitions, self.mask)
                    lvl.rules_report = lvl.model.verify(transitions, self.mask)
                    if lvl.breaks >= MAX_BREAKS_PER_LEVEL or \
                            len(lvl.model.verified_moves()) < MIN_VERIFIED_MOVES:
                        self._mark_fallback(lvl, "prediction-breaks")
                    self._flush(state_path)
                    return TurnResult(step_executed=True)
        if clean and len(actions) > executed and executed >= PLAN_BATCH:
            lvl.active_plan = (target_desc, actions[executed:], probe_last)
        elif clean and probe_last and executed == len(actions):
            # probe executed: absorb its outcome into the model right away so
            # the next BFS sees the new permeable color or the new blocker.
            if lvl.pending_probe is not None and not payload.get("board_changed"):
                if lvl.pending_probe not in lvl.refused_probes:
                    lvl.refused_probes.append(lvl.pending_probe)
            lvl.pending_probe = None
            cur, history = read_state(state_path)
            self._feed_mask(history)
            transitions = [t for t in transitions_from_history(history)
                           if t.level_before == lvl.level]
            lvl.model.mine(transitions, self.mask)
            lvl.rules_report = lvl.model.verify(transitions, self.mask)
            events = lvl.model.unexplained + lvl.breaks
            if events != lvl.events_mark:
                # a world event just happened: re-arm probes NOW (not at
                # exhaustion) so the refused-first retry lands inside the
                # same game round as the event that may have unlocked it.
                lvl.events_mark = events
                lvl.tried_targets = [k for k in lvl.tried_targets
                                     if k[0] != "probe"]
                self.log(f"level={lvl.level} probes-rearmed-now events={events}")
        if lvl.plans_run >= MAX_PLANS_PER_LEVEL:
            self._mark_fallback(lvl, "plan-budget-exhausted")
        self._flush(state_path)
        return TurnResult(step_executed=executed > 0)

    def _next_plan(self, lvl, grid, pos, parents):
        """Pick the next untried goal; then coverage sweep; then a frontier
        probe. Returns (desc, actions, probe_last) or None."""
        model = lvl.model
        for color, cells in goal_targets(model, grid, self.mask, pos):
            touching = positions_touching(model, cells, parents)
            for goal in touching:
                key = ("color", color, goal)
                if key in lvl.tried_targets:
                    continue
                actions = path_to(parents, goal)
                if actions:
                    lvl.tried_targets.append(key)
                    return f"color={color}@{goal}", actions, False
        # coverage sweep: visit the NEAREST reachable position never stood on.
        # BFS insertion order == distance order, so the first unvisited entry
        # is the nearest; repeating this tours the whole reachable set. With
        # eps=0.17 actions are nearly free, so the tour costs ~nothing and
        # guarantees the sprite touches every reachable cell of the level.
        if lvl.novelty_run < MAX_SWEEP_PLANS:
            for goal in parents:
                if goal == pos or goal in lvl.visited:
                    continue
                actions = path_to(parents, goal)
                if actions:
                    lvl.novelty_run += 1
                    return f"sweep@{goal}", actions, False
        # frontier probe: one cheap experiment past the verified boundary.
        # Previously REFUSED probes come first: after a world event they are
        # the locked doors most likely to have just opened, and retrying them
        # immediately beats re-touring unprobed cells (the ls20 key/door pair
        # is lost to the round timer otherwise).
        cands = frontier_probes(model, grid, parents)
        refused = [c for c in cands if ("probe", c[1], c[2]) in lvl.refused_probes]
        fresh = [c for c in cands if ("probe", c[1], c[2]) not in lvl.refused_probes]
        for _, ppos, action, colors in refused + fresh:
            key = ("probe", ppos, action)
            if key in lvl.tried_targets:
                continue
            base = path_to(parents, ppos)
            if base is None:
                continue
            lvl.tried_targets.append(key)
            lvl.pending_probe = key
            return f"probe{list(colors)}@{ppos}+{action}", base + [action], True
        # The world is STATEFUL: an unexplained event (a pickup, a toggle, a
        # break) can change what a refused move now does -- the ls20 key/door
        # is exactly this shape. Each NEW unexplained event re-arms every
        # probe once; with no new events, exhaustion is final.
        events = lvl.model.unexplained + lvl.breaks
        if events != lvl.events_mark:
            lvl.events_mark = events
            before = len(lvl.tried_targets)
            lvl.tried_targets = [k for k in lvl.tried_targets if k[0] != "probe"]
            self.log(f"level={lvl.level} probes-rearmed events={events} "
                     f"(cleared {before - len(lvl.tried_targets)})")
            return self._next_plan(lvl, grid, pos, parents)
        return None

    def _flush(self, state_path):
        try:
            self.report_cb(self.report())
        except Exception:
            pass


# ===========================================================================
# the analyzer wrapper (what the patched solver constructs)
# ===========================================================================
class ExecWMAnalyzer:
    """Wraps the stock ToolAgent. Deterministic exec-WM turns run with ZERO
    LLM round-trips; fallback turns delegate to the stock agent untouched."""

    def __init__(self, inner, game=None, index=0, solver=None):
        self.inner = inner
        self.game = game
        self.index = index
        self.solver = solver
        game_id = getattr(getattr(game, "game_run", None), "game_id", None) \
            or getattr(game, "game_id", None) or getattr(game, "env_name", None) \
            or f"game{index}"
        self._log_lock = threading.Lock()
        self._transcript_path = None
        llm_cfg = self._llm_config(inner)
        self.controller = ExecWMController(
            str(game_id), self._log, self._write_report, llm_cfg=llm_cfg)
        self._log(f"armed {EXECWM_VERSION} game={game_id} "
                  f"llm={'on' if llm_cfg else 'off'}")

    # ---- token accounting: the solver reads .generated_tokens ----
    @property
    def generated_tokens(self):
        inner_tokens = getattr(self.inner, "generated_tokens", None)
        if inner_tokens is None:
            inner_tokens = getattr(self.inner, "total_tokens", 0)
        return int(inner_tokens or 0) + int(self.controller.llm_tokens)

    def _llm_config(self, inner):
        if int(os.environ.get("ARC3_EXECWM_LLM", "1") or "1") == 0:
            return None
        model = getattr(inner, "_model", None)
        base_url = getattr(model, "base_url", None)
        model_id = getattr(model, "model_id", None)
        if not base_url or not model_id:
            return None
        return {"base_url": str(base_url), "model": str(model_id),
                "api_key": getattr(inner, "_api_key", "") or ""}

    def _log(self, msg):
        line = f"[execwm] {msg}"
        with self._log_lock:
            print(line, flush=True)
            tp = self._transcript_path
            if tp is not None:
                try:
                    with open(tp, "a", encoding="utf-8") as f:
                        f.write(line + "\n")
                except Exception:
                    pass

    def _write_report(self, report):
        job_dir = getattr(self.solver, "job_dir", None)
        if not job_dir:
            return
        try:
            from pathlib import Path
            out_dir = Path(job_dir) / "execwm"
            out_dir.mkdir(parents=True, exist_ok=True)
            stem = "".join(ch if ch.isalnum() or ch in "-_" else "_"
                           for ch in report["game_id"])
            out = out_dir / f"{stem}_p{self.index}.json"
            tmp = out.with_suffix(".json.tmp")
            tmp.write_text(json.dumps(report, indent=1), encoding="utf-8")
            tmp.replace(out)
        except Exception:
            pass

    # ---- the analyzer protocol ----
    def analyze(self, state_path, action_num, valid_actions=None, step_env=None,
                transcript_path=None, analysis_step=None, transcript_updated=None,
                request_timeout_seconds=None, should_stop=None, **kwargs):
        self._transcript_path = transcript_path
        current, _ = read_state(state_path) if state_path.exists() else (None, [])
        use_wm = (current is not None and current.get("grid")
                  and step_env is not None
                  and self.controller.wants_turn(current, valid_actions))
        if use_wm:
            try:
                result = self.controller.run_turn(
                    state_path, step_env, should_stop, valid_actions)
                if result.step_executed:
                    return result
                # controller did no work this turn (e.g. just fell back):
                # fall through to the stock agent in the same turn.
                current, _ = read_state(state_path)
                lvl = self.controller.levels.get(current["level"]) if current else None
                if lvl is not None and not lvl.fallback:
                    return result
            except Exception as exc:  # never let exec-WM kill the run
                self._log(f"controller-error {type(exc).__name__}: {exc} -> fallback")
                try:
                    lvl = self.controller._lvl(current["level"])
                    self.controller._mark_fallback(lvl, f"controller-error:{type(exc).__name__}")
                except Exception:
                    self.controller.disabled_reason = "controller-error"
        return self.inner.analyze(
            state_path, action_num, valid_actions=valid_actions, step_env=step_env,
            transcript_path=transcript_path, analysis_step=analysis_step,
            transcript_updated=transcript_updated,
            request_timeout_seconds=request_timeout_seconds,
            should_stop=should_stop, **kwargs)


def maybe_wrap_analyzer(inner, game=None, index=0, solver=None):
    """Called from the patched HarnessSolver._make_analyzer. Honors the arm
    kill-switch: ARC3_EXECWM=0 leaves the stock analyzer untouched."""
    if int(os.environ.get("ARC3_EXECWM", "1") or "1") == 0:
        return inner
    return ExecWMAnalyzer(inner, game=game, index=index, solver=solver)
'''

assert hashlib.sha256(_EWM_SOURCE.encode("utf-8")).hexdigest()[:16] == _EWM_SHA, \
    "EXECWM FATAL: embedded module bytes do not match the build-time sha"

_ewm_src_root = None
for _cand in Path("/kaggle/input").rglob("taaf-kaggle-bundle.json"):
    _c = _cand.parent / "src" / "ARC3-Inference"
    if _c.is_dir():
        _ewm_src_root = _c
        break
assert _ewm_src_root is not None, "EXECWM FATAL: bundle ARC3-Inference not found"

_ewm_dst = Path("/kaggle/working/execwm_patched/ARC3-Inference")
if _ewm_dst.exists():
    shutil.rmtree(_ewm_dst)
shutil.copytree(_ewm_src_root, _ewm_dst)

(_ewm_dst / "inference" / "agent" / "exec_wm.py").write_text(_EWM_SOURCE, encoding="utf-8")

_ewm_solver = _ewm_dst / "inference" / "framework" / "solver.py"
_ewm_anchor = '    def _make_analyzer(\n'
_ewm_wrap = '    def _make_analyzer(self, game, index, local_server=None):\n        inner = self._make_analyzer_stock(game, index, local_server)\n        try:\n            from inference.agent.exec_wm import maybe_wrap_analyzer\n            return maybe_wrap_analyzer(inner, game=game, index=index, solver=self)\n        except Exception as exc:\n            print(f"[execwm] wrap-failed {type(exc).__name__}: {exc} -- "\n                  "stock analyzer in use", flush=True)\n            return inner\n\n    def _make_analyzer_stock(\n'
_ewm_text = _ewm_solver.read_text(encoding="utf-8")
_ewm_n = _ewm_text.count(_ewm_anchor)
assert _ewm_n == 1, f"EXECWM FATAL: solver anchor x{_ewm_n} (vehicle drifted)"
assert "_make_analyzer_stock" not in _ewm_text, "EXECWM FATAL: double apply"
_ewm_solver.write_text(_ewm_text.replace(_ewm_anchor, _ewm_wrap, 1), encoding="utf-8")

sys.path.insert(0, str(_ewm_dst))
import inference.framework.solver as _ewm_chk
assert str(_ewm_dst) in str(Path(_ewm_chk.__file__)), \
    f"EXECWM FATAL: wrong module resolved: {_ewm_chk.__file__}"

os.environ["ARC3_EXECWM"] = "1"
os.environ.setdefault("ARC3_EXECWM_LLM", "1")
print(f"[execwm] patch applied sha={_EWM_SHA} shadowed at {_ewm_dst}", flush=True)


In [ ]:
def _soft_end_time(max_runtime_s: float, *, run_as_submission: bool) -> datetime | None:
    if run_as_submission or max_runtime_s <= 0:
        return None
    budget = max(1.0, max_runtime_s)
    buffer = min(SOFT_DEADLINE_BUFFER_S, budget / 2)
    start = datetime.fromtimestamp(NOTEBOOK_START_EPOCH)
    return start + timedelta(seconds=budget - buffer)


def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ.get("ARC_BASE_URL", "http://gateway:8001/"),
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


@contextlib.contextmanager
def _tee_to_file(log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_file = open(log_path, "w", buffering=1)
    original_stdout = sys.stdout
    original_stderr = sys.stderr
    sys.stdout = _Tee(original_stdout, log_file)
    sys.stderr = _Tee(original_stderr, log_file)
    try:
        yield
    finally:
        sys.stdout = original_stdout
        sys.stderr = original_stderr
        log_file.close()


class _Tee:
    def __init__(self, *streams: TextIO) -> None:
        self._streams = streams

    def write(self, data: str) -> int:
        n = 0
        for stream in self._streams:
            n = stream.write(data)
        return n

    def flush(self) -> None:
        for stream in self._streams:
            stream.flush()

    def isatty(self) -> bool:
        return any(getattr(stream, "isatty", lambda: False)() for stream in self._streams)

In [ ]:
true_submission = _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
run_as_submission = _env_bool("TAAF_RUN_AS_SUBMISSION", False) or true_submission
os.environ["ONLY_RESET_LEVELS"] = "true"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if run_as_submission else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if run_as_submission else "0"

with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = run_as_submission
target.is_competition_rerun = true_submission
soft_end = _soft_end_time(float(getattr(target, "max_runtime_s", 0.0) or 0.0), run_as_submission=run_as_submission)

with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

In [ ]:
# Inline customization hook — Q38 P1-style public evaluation.
#
# Q38 P1 runs the full 25 public ARC-AGI-3 games once each (25 games × 1 pass).
# This override applies only to the public/offline notebook run. Competition reruns
# still replace bm.games from Kaggle's live gateway in the final run cell.

print("Benchmark analyzer model:", os.environ.get("INFERENCE_ANALYZER_MODEL"))
print("Qwen3.8 model path:", os.environ.get("TAAF_QWEN_MODEL_PATH"))

Q38_P1_PUBLIC_GAME_IDS = [
    "ar25-0c556536",
    "bp35-0a0ad940",
    "cd82-fb555c5d",
    "cn04-2fe56bfb",
    "dc22-fdcac232",
    "ft09-0d8bbf25",
    "g50t-5849a774",
    "ka59-38d34dbb",
    "lf52-271a04aa",
    "lp85-305b61c3",
    "ls20-9607627b",
    "m0r0-492f87ba",
    "r11l-495a7899",
    "re86-8af5384d",
    "s5i5-18d95033",
    "sb26-7fbdac44",
    "sc25-635fd71a",
    "sk48-d8078629",
    "sp80-589a99af",
    "su15-1944f8ab",
    "tn36-ef4dde99",
    "tr87-cd924810",
    "tu93-0768757b",
    "vc33-5430563c",
    "wa30-ee6fef47",
]

if not true_submission:
    if len(Q38_P1_PUBLIC_GAME_IDS) != 25 or len(set(Q38_P1_PUBLIC_GAME_IDS)) != 25:
        raise RuntimeError("Q38 P1 public game list must contain exactly 25 unique games.")
    if not bm.games:
        raise RuntimeError("benchmark_initial.pkl contains no template public game.")

    import taaf.game_api

    template_game = bm.games[0]
    arcade_spec = getattr(template_game, "arcade_spec", None)
    if arcade_spec is None:
        arcade_spec = getattr(template_game, "_arcade_spec", None)
    if arcade_spec is None:
        raise RuntimeError(
            "Could not recover the public ArcadeSpec from benchmark_initial.pkl; "
            "cannot construct the 25-game Q38 P1 evaluation set."
        )

    bm.games = [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=arcade_spec)
        for game_id in Q38_P1_PUBLIC_GAME_IDS
    ]
    bm.n_passes = 1
    bm.game_weights = None

    # These already match Q38 P1 in the source notebook; set them explicitly so the
    # intended evaluation configuration is visible and stable.
    if hasattr(bm.solver, "concurrency"):
        bm.solver.concurrency = 28
    if hasattr(bm.solver, "max_runtime_s_per_game"):
        bm.solver.max_runtime_s_per_game = 7920.0

    bm.label = f"{bm.label}-25g-p1"
    print(f"Public evaluation override: {len(bm.games)} games × {bm.n_passes} pass = {len(bm.games) * bm.n_passes} runs")
    print("Public evaluation concurrency:", getattr(bm.solver, "concurrency", None))
    print("Public per-game runtime cap (s):", getattr(bm.solver, "max_runtime_s_per_game", None))


In [ ]:
run_context = contextlib.nullcontext() if run_as_submission else _tee_to_file(WORKING_DIR / "stdout.log")
with run_context:
    preamble = (BUNDLE_DIR / "preamble.txt").read_text(encoding="utf-8")
    print(preamble)
    print(f"deploy.kaggle: working_dir             = {WORKING_DIR}")
    print(f"deploy.kaggle: run_as_submission       = {run_as_submission}")
    print(f"deploy.kaggle: competition_rerun       = {true_submission}")
    print(f"deploy.kaggle: soft_end_time           = {soft_end}")
    print("---")

    bundled_git_status = BUNDLE_DIR / "git_status.txt"
    if bundled_git_status.is_file():
        (WORKING_DIR / "git_status.txt").write_text(
            bundled_git_status.read_text(encoding="utf-8"),
            encoding="utf-8",
        )

    if true_submission:
        # Competition reruns use Kaggle's live gateway instead of the bundled offline games.
        os.environ.setdefault("ARC_API_KEY", "test-key-123")
        os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
        os.environ.setdefault("SCHEME", "http")
        os.environ.setdefault("HOST", "gateway")
        os.environ.setdefault("PORT", "8001")
        os.environ.setdefault("OPERATION_MODE", "competition")
        os.environ.setdefault("ENVIRONMENTS_DIR", "")
        os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

        deadline = time.monotonic() + 600.0
        last_error = ""
        while time.monotonic() < deadline:
            try:
                with urlopen("http://gateway:8001/api/games", timeout=10) as response:
                    if response.status < 500:
                        break
            except Exception as exc:
                last_error = repr(exc)
            time.sleep(5)
        else:
            raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")

        bm.games = _competition_games()
        bm.n_passes = 1
        bm.game_weights = None

    try:
        await bm.run(
            soft_end_time=soft_end,
            runtime_environment=target,
            minimal_diagnostics=run_as_submission,
        )
        if not true_submission and Path("/kaggle/input").exists():
            try:
                import pandas as pd

                submission = pd.DataFrame(
                    data=[["1_0", "1", True, 1]],
                    columns=["row_id", "game_id", "end_of_game", "score"],
                )
                submission.to_parquet(WORKING_DIR / "submission.parquet", index=False)
            except Exception as exc:
                print(f"taaf.kaggle: could not write offline dummy submission: {exc!r}", flush=True)
    finally:
        _run_shell_commands("teardown_commands.json", label="teardown", check=False)

In [ ]:
from html import escape

from IPython.display import HTML, display

diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    # Isolate the full document in an iframe so its styles don't leak into the notebook.
    display(
        HTML(
            f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
            'width="100%" height="900" style="border:0"></iframe>'
        )
    )
else:
    print("No diagnostics.html — minimal diagnostics (real submission) suppresses it.")